In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

movies = pd.read_csv('/content/drive/MyDrive/movie-recommendation/movies.csv')
ratings = pd.read_csv('/content/drive/MyDrive/movie-recommendation/ratings.csv')

print(f"Movies: {movies.shape}")
print(f"Ratings: {ratings.shape}")

Movies: (62423, 3)
Ratings: (25000095, 4)


In [ ]:
# Filter only popular movies (rated by at least 50 users)
movie_rating_counts = ratings.groupby('movieId')['rating'].count()
popular_movie_ids = movie_rating_counts[movie_rating_counts >= 50].index

ratings_popular = ratings[ratings['movieId'].isin(popular_movie_ids)]

print(f"Popular movies count: {len(popular_movie_ids)}")
print(f"Ratings after filter: {len(ratings_popular)}")

Popular movies count: 13176
Ratings after filter: 24644928


In [ ]:
movies_with_ratings = ratings_popular.merge(movies, on='movieId')
print(movies_with_ratings.shape)
movies_with_ratings.head()


(24644928, 6)


,userId,movieId,rating,timestamp,title,genres
0,1,296,5.0,1147880044,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
1,1,306,3.5,1147868817,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama
2,1,307,5.0,1147868828,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama
3,1,665,5.0,1147878820,Underground (1995),Comedy|Drama|War
4,1,899,3.5,1147868510,Singin' in the Rain (1952),Comedy|Musical|Romance


In [ ]:
# Keep only users who rated at least 50 movies
active_users = movies_with_ratings.groupby('userId')['rating'].count()
active_user_ids = active_users[active_users >= 50].index

movies_with_ratings_filtered = movies_with_ratings[
    movies_with_ratings['userId'].isin(active_user_ids)
]

print(f"Active users: {len(active_user_ids)}")
print(f"Ratings remaining: {len(movies_with_ratings_filtered)}")

Active users: 102293
Ratings remaining: 22749997


In [ ]:
from scipy.sparse import csr_matrix

# Create mappings for users and movies
user_ids = movies_with_ratings_filtered['userId'].astype('category')
movie_titles = movies_with_ratings_filtered['title'].astype('category')

# Build sparse matrix
sparse_matrix = csr_matrix((
    movies_with_ratings_filtered['rating'].values,
    (user_ids.cat.codes, movie_titles.cat.codes)
))

print(f"Sparse matrix shape: {sparse_matrix.shape}")
print(f"Memory usage: {sparse_matrix.data.nbytes / 1024 / 1024:.1f} MB")

# Save category mappings for lookup later
user_index = dict(enumerate(user_ids.cat.categories))
movie_index = dict(enumerate(movie_titles.cat.categories))
movie_to_idx = {v: k for k, v in movie_index.items()}

Sparse matrix shape: (102293, 13173)
Memory usage: 173.6 MB


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend(movie_name, n=10):
    if movie_name not in movie_to_idx:
        return "Movie not found."

    idx = movie_to_idx[movie_name]

    movie_vector = sparse_matrix.T[idx]
    similarity_scores = cosine_similarity(movie_vector, sparse_matrix.T).flatten()

    similar_indices = similarity_scores.argsort()[::-1][1:n+1]

    results = []
    for i in similar_indices:
        results.append({
            'title': movie_index[i],
            'similarity': round(similarity_scores[i], 3)
        })

    return pd.DataFrame(results)

def find_movie_name(user_input):
    user_input = user_input.lower()
    return [title for title in movie_to_idx.keys() if user_input in title.lower()]

# Test
print(find_movie_name("dark knight"))

['Dark Knight Rises, The (2012)', 'Dark Knight, The (2008)']


In [ ]:
recommend("Dark Knight, The (2008)")

,title,similarity
0,Inception (2010),0.745
1,Batman Begins (2005),0.704
2,Iron Man (2008),0.699
3,"Dark Knight Rises, The (2012)",0.685
4,WALL·E (2008),0.650
5,"Lord of the Rings: The Return of the King, The...",0.645
6,Inglourious Basterds (2009),0.644
7,"Prestige, The (2006)",0.633
8,"Departed, The (2006)",0.626
9,Fight Club (1999),0.626


In [ ]:
# Create genre similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Clean genres (replace | with space so TF-IDF can read it)
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

# Build TF-IDF matrix on genres
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

# Map movieId to index
movies = movies.reset_index(drop=True)
movie_id_to_idx = {row['movieId']: idx for idx, row in movies.iterrows()}
title_to_movie_id = dict(zip(movies['title'], movies['movieId']))

print("Genre matrix shape:", tfidf_matrix.shape)

Genre matrix shape: (62423, 24)


In [ ]:
def hybrid_recommend(movie_name, n=10, rating_weight=0.7, genre_weight=0.3):
    if movie_name not in movie_to_idx:
        return "Movie not found."

    # Rating based similarity
    idx = movie_to_idx[movie_name]
    movie_vector = sparse_matrix.T[idx]
    rating_scores = cosine_similarity(movie_vector, sparse_matrix.T).flatten()

    # Genre based similarity
    if movie_name in title_to_movie_id:
        movie_id = title_to_movie_id[movie_name]
        genre_idx = movie_id_to_idx[movie_id]
        genre_scores_raw = linear_kernel(tfidf_matrix[genre_idx], tfidf_matrix).flatten()
    else:
        genre_scores_raw = [0] * tfidf_matrix.shape[0]

    # Map genre scores to same index as rating scores
    results = []
    similar_indices = rating_scores.argsort()[::-1][1:n+1]

    for i in similar_indices:
        title = movie_index[i]
        r_score = rating_scores[i]

        # Get genre score for this title
        g_score = 0
        if title in title_to_movie_id:
            mid = title_to_movie_id[title]
            gidx = movie_id_to_idx[mid]
            g_score = genre_scores_raw[gidx]

        final_score = (rating_weight * r_score) + (genre_weight * g_score)
        results.append({
            'title': title,
            'final_score': round(final_score, 3),
            'rating_sim': round(r_score, 3),
            'genre_sim': round(g_score, 3)
        })

    results = sorted(results, key=lambda x: x['final_score'], reverse=True)
    return pd.DataFrame(results)

# Test
hybrid_recommend("Dark Knight, The (2008)")

,title,final_score,rating_sim,genre_sim
0,Batman Begins (2005),0.785,0.704,0.975
1,Inception (2010),0.747,0.745,0.751
2,"Dark Knight Rises, The (2012)",0.746,0.685,0.888
3,Fight Club (1999),0.592,0.626,0.515
4,"Departed, The (2006)",0.550,0.626,0.373
5,Iron Man (2008),0.537,0.699,0.160
6,Inglourious Basterds (2009),0.533,0.644,0.274
7,"Lord of the Rings: The Return of the King, The...",0.523,0.645,0.239
8,"Prestige, The (2006)",0.460,0.633,0.055
9,WALL·E (2008),0.455,0.650,0.000


In [ ]:
import pickle

# Save everything Streamlit will need
with open('/content/drive/MyDrive/movie-recommendation/model.pkl', 'wb') as f:
    pickle.dump({
        'sparse_matrix': sparse_matrix,
        'movie_index': movie_index,
        'movie_to_idx': movie_to_idx,
        'title_to_movie_id': title_to_movie_id,
        'movie_id_to_idx': movie_id_to_idx,
        'tfidf_matrix': tfidf_matrix,
        'movies': movies
    }, f)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
user_input = input("Enter movie name: ")

matches = find_movie_name(user_input)

if len(matches) == 0:
    print("No matching movie found.")

elif len(matches) == 1:
    print("Using:", matches[0])
    print(hybrid_recommend(matches[0]))

else:
    print("\nYou searched:", user_input)
    print("Did you mean:\n")

    for i, movie in enumerate(matches[:10]):
        print(f"{i+1}. {movie}")

    choice_input = input("\nEnter number: ")

    if not choice_input.isdigit():
        print("Please enter a valid number")
    else:
        choice = int(choice_input)

        if choice < 1 or choice > min(10, len(matches)):
            print("Invalid choice")
        else:
            selected_movie = matches[choice - 1]
            print("\nRecommendations for:", selected_movie)
            print(hybrid_recommend(selected_movie))

Enter movie name: fight club
Using: Fight Club (1999)
                                               title  final_score  rating_sim  \
0                           Kill Bill: Vol. 1 (2003)        0.737       0.647   
1                                Pulp Fiction (1994)        0.704       0.677   
2                          American History X (1998)        0.657       0.650   
3                                 Matrix, The (1999)        0.652       0.735   
4                                   Gladiator (2000)        0.577       0.636   
5                                     Memento (2000)        0.574       0.691   
6  Lord of the Rings: The Return of the King, The...        0.557       0.648   
7                             American Beauty (1999)        0.523       0.675   
8  Lord of the Rings: The Fellowship of the Ring,...        0.470       0.672   
9      Lord of the Rings: The Two Towers, The (2002)        0.455       0.649   

   genre_sim  
0      0.946  
1      0.765  
2      0.

In [ ]:
print(movies[['title', 'genres']].head(10))


                                title  \
0                    Toy Story (1995)   
1                      Jumanji (1995)   
2             Grumpier Old Men (1995)   
3            Waiting to Exhale (1995)   
4  Father of the Bride Part II (1995)   
5                         Heat (1995)   
6                      Sabrina (1995)   
7                 Tom and Huck (1995)   
8                 Sudden Death (1995)   
9                    GoldenEye (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
5                        Action|Crime|Thriller  
6                               Comedy|Romance  
7                           Adventure|Children  
8                                       Action  
9                    Action|Adventure|Thriller  
